In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# metrics.py lives at ../../../src/metrics/metrics.py relative to this notebook
sys.path.insert(0, str((Path("../../../src/metrics")).resolve()))
from metrics import (
    organize_multiverse_results,
    within_bucket_variation,
    total_variance_decomposition,
)

In [ ]:
DIMS = [
    "death_outliers",
    "damage_outliers",
    "femininity_calculation",
    "damage_transform",
    "model",
    "main_interaction",
    "other_predictors",
    "covariates",
]
METRIC = "mean_deaths"


def load_df(path):
    df = pd.read_csv(path)
    df = df[df["mv_error_type"].isna()].copy()
    return df.rename(columns={f"mv_dim_{d}": d for d in DIMS})


def compute_importances(df):
    """
    Returns (wbv_dict, tvd_dict) keyed by dimension name.

    wbv values: mean decrease in variance at each depth (normalized to sum=1).
    tvd values: expected_var_within_groups / var_total (NaN if the constrained
      main_interaction × other_predictors pair violates the full-factorial
      assumption required by total_variance_decomposition).
    """
    organized = organize_multiverse_results(df, DIMS, [METRIC])

    # within_bucket_variation returns {depth: importance}; map to dim names
    wbv_raw = within_bucket_variation(organized, METRIC)
    wbv = {DIMS[depth]: val for depth, val in wbv_raw.items()}
    total = sum(wbv.values()) or 1
    wbv = {dim: val / total for dim, val in wbv.items()}

    # total_variance_decomposition requires a full-factorial design;
    # the main_interaction × other_predictors constraint breaks this in
    # any analysis where both those dims are free.
    try:
        tvd_raw = total_variance_decomposition(organized, METRIC)
        var_total = tvd_raw[DIMS[0]]["var_total"]  # same for all dims
        tvd = {
            dim: tvd_raw[dim]["expected_var_within_groups"] / var_total
            for dim in DIMS
        }
    except AssertionError:
        tvd = {dim: np.nan for dim in DIMS}

    return wbv, tvd


# --- load and compute ---
# key: short label used as column header in the heatmaps
results = {}  # label -> (wbv_dict, tvd_dict)

results["full"] = compute_importances(
    load_df(Path("../output/runs/1/data/agg_1_run_outputs.csv.gz"))
)

for dim in DIMS:
    path = Path(f"{dim}/output/runs/1/data/agg_1_run_outputs.csv.gz")
    results[f"fix:\n{dim}"] = compute_importances(load_df(path))

print("Analyses computed:", list(results.keys()))

In [ ]:
# --- build DataFrames for heatmaps ---
labels = list(results.keys())

wbv_df = pd.DataFrame(
    {label: results[label][0] for label in labels}, index=DIMS
)
tvd_df = pd.DataFrame(
    {label: results[label][1] for label in labels}, index=DIMS
)


def plot_heatmap(df, title, fmt=".2f", cmap="YlOrRd"):
    fig, ax = plt.subplots(figsize=(13, 4.5))
    vals = df.values.astype(float)

    # mask NaNs so they render as a distinct color
    masked = np.ma.masked_invalid(vals)
    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad(color="#cccccc")

    im = ax.imshow(masked, aspect="auto", cmap=cmap_obj,
                   vmin=np.nanmin(vals), vmax=np.nanmax(vals))
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)

    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, fontsize=8)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=9)
    ax.set_title(title, fontsize=11, pad=10)

    # annotate cells
    for r in range(vals.shape[0]):
        for c in range(vals.shape[1]):
            v = vals[r, c]
            text = f"{v:{fmt}}" if not np.isnan(v) else "N/A"
            ax.text(c, r, text, ha="center", va="center",
                    fontsize=7, color="black")

    fig.tight_layout()
    plt.show()


plot_heatmap(
    wbv_df,
    title=(
        "Within-bucket variation importance (relative, sums to 1 per column)\n"
        "Rows = dimension, Columns = which dimension is fixed (or 'full' for the unablated multiverse)"
    ),
)

plot_heatmap(
    tvd_df,
    title=(
        "Variance decomposition importance = E[Var(Y|others)] / Var(Y)  (fraction of total variance)\n"
        "N/A: main_interaction × other_predictors constraint violates full-factorial assumption"
    ),
    fmt=".3f",
)